## **⭐ The 4 Join Types and the Settings That Control Them**

**BROADCAST → BroadcastHashJoin → fastest**
 - Best when one side is small (≤ 25 MB in Fabric).  
 - Avoids shuffles.  
 - Uses in‑memory hash tables on each executor.

**MERGE → SortMergeJoin → second fastest**
 - Best when both sides are large.  
 - Requires shuffle + sort, but is memory‑safe and highly scalable.  
 - Spark’s preferred fallback when broadcast is not possible.

**SHUFFLE_HASH → ShuffledHashJoin → situational**
 - Can outperform SMJ only when:  
 - both sides are already hash‑partitioned on the join key  
 - data is evenly distributed  

Risky: can cause OOM under skew.  
AQE rarely chooses it unless conditions are perfect.

**SHUFFLE_REPLICATE_NL → NestedLoopJoin / Cartesian → slowest**

Used for:  
 - non‑equijoins (>, <, BETWEEN)  
 - missing join keys  
 - Extremely expensive because it replicates one side to all partitions.


#### **Getting Demo Environment Ready**

In [ ]:
SCHEMA = "optimization"

spark.sql(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE")

<u>**Create a new schema**</u>

In [6]:
SCHEMA = "optimization"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA} COMMENT 'Demo schema for Spark physical joins'")

spark.sql(f"USE {SCHEMA}")
# spark.catalog.setCurrentDatabase("sales_demo")


StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 9, Finished, Available, Finished, False)

DataFrame[]

<u>**Set Schema Context**</u>

In [4]:
print(spark.catalog.currentDatabase())
spark.sql("SHOW TABLES").show()

StatementMeta(, 6ea3545c-5a40-406a-b081-502cd08ea53d, 6, Finished, Available, Finished, False)

chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



<u>**Generating Dummy Data**</u>

In [5]:
# ================================================================
# Spark Physical Joins Demo Dataset Generator
# For: Microsoft Fabric Lakehouse
# Purpose: Demonstrate Broadcast Hash Join, Sort Merge Join,
#          Shuffle Hash Join, Broadcast Nested Loop Join,
#          and Bucketed Joins
# ================================================================
# Run this in a Microsoft Fabric Notebook attached to a Lakehouse.
# ================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# ----------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------
LAKEHOUSE_PATH   = "Tables"          # Fabric Lakehouse managed tables path
FACT_ROWS        = 50_000_000        # 50 million rows
SMALL_DIM_ROWS   = 500               # < 10 MB  -> Broadcast Hash Join demo
PRODUCT_DIM_ROWS = 10_000            # < 10 MB  -> Broadcast Hash Join demo
DATE_DIM_DAYS    = 1_826             # ~5 years -> Broadcast + Partition Pruning demo
LARGE_DIM_ROWS   = 20_000_000        # ~1.5 GB  -> Sort Merge Join demo


# ================================================================
# STEP 1: Create the SMALL DIMENSION table (dim_store)
#         Target size: < 10 MB  (Broadcast Hash Join candidate)
# ================================================================
print("STEP 1: Creating dim_store (small dimension < 10 MB)")

dim_store = (
    spark.range(0, SMALL_DIM_ROWS)
         .withColumnRenamed("id", "store_id")
         .withColumn("store_name",   F.concat(F.lit("Store_"), F.col("store_id")))
         .withColumn("region",       F.expr("CASE WHEN store_id % 4 = 0 THEN 'North' "
                                            "     WHEN store_id % 4 = 1 THEN 'South' "
                                            "     WHEN store_id % 4 = 2 THEN 'East'  "
                                            "     ELSE 'West' END"))
         .withColumn("country",      F.expr("CASE WHEN store_id % 3 = 0 THEN 'USA' "
                                            "     WHEN store_id % 3 = 1 THEN 'Canada' "
                                            "     ELSE 'Mexico' END"))
         .withColumn("store_type",   F.expr("CASE WHEN store_id % 2 = 0 THEN 'Retail' ELSE 'Wholesale' END"))
)

(dim_store
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dim_store"))
    

print(f"dim_store rows: {spark.table('dim_store').count():,}")


# ================================================================
# STEP 2A: Create the PRODUCT DIMENSION table (dim_product)
#         Target size: < 10 MB  (Broadcast Hash Join candidate)
# ================================================================
print("STEP 2A: Creating dim_product (small dimension < 10 MB)")

dim_product = (
    spark.range(0, PRODUCT_DIM_ROWS)
         .withColumnRenamed("id", "product_id")
         .withColumn("product_name",   F.concat(F.lit("Product_"), F.col("product_id")))
         .withColumn("category",       F.expr("CASE WHEN product_id % 6 = 0 THEN 'Electronics' "
                                              "     WHEN product_id % 6 = 1 THEN 'Furniture' "
                                              "     WHEN product_id % 6 = 2 THEN 'Clothing' "
                                              "     WHEN product_id % 6 = 3 THEN 'Grocery' "
                                              "     WHEN product_id % 6 = 4 THEN 'Toys' "
                                              "     ELSE 'Sports' END"))
         .withColumn("sub_category",   F.expr("CASE WHEN product_id % 4 = 0 THEN 'Premium' "
                                              "     WHEN product_id % 4 = 1 THEN 'Standard' "
                                              "     WHEN product_id % 4 = 2 THEN 'Budget' "
                                              "     ELSE 'Clearance' END"))
         .withColumn("brand",          F.concat(F.lit("Brand_"), F.expr("cast(product_id % 100 as int)")))
         .withColumn("supplier",       F.concat(F.lit("Supplier_"), F.expr("cast(product_id % 50 as int)")))
         .withColumn("list_price",     F.expr("round(rand(48) * 500 + 10, 2)"))
         .withColumn("cost_price",     F.expr("round(list_price * 0.6, 2)"))
         .withColumn("is_active",      F.expr("CASE WHEN product_id % 10 = 0 THEN false ELSE true END"))
)

(dim_product
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dim_product"))

print(f"dim_product rows: {spark.table('dim_product').count():,}")


# ================================================================
# STEP 2B: Create the DATE DIMENSION table (dim_date)
#         Target size: < 10 MB  (Broadcast + Partition Pruning demo)
# ================================================================
print("STEP 2B: Creating dim_date (date dimension < 10 MB)")

dim_date = (
    spark.range(0, DATE_DIM_DAYS)
         .withColumnRenamed("id", "date_key")
         .withColumn("full_date",     F.expr("date_add('2022-01-01', cast(date_key as int))"))
         .withColumn("year",          F.year("full_date"))
         .withColumn("quarter",       F.quarter("full_date"))
         .withColumn("month",         F.month("full_date"))
         .withColumn("month_name",    F.date_format("full_date", "MMMM"))
         .withColumn("day",           F.dayofmonth("full_date"))
         .withColumn("day_of_week",   F.dayofweek("full_date"))
         .withColumn("day_name",      F.date_format("full_date", "EEEE"))
         .withColumn("week_of_year",  F.weekofyear("full_date"))
         .withColumn("is_weekend",    F.expr("CASE WHEN dayofweek(full_date) IN (1,7) THEN true ELSE false END"))
         .withColumn("fiscal_year",   F.expr("CASE WHEN month(full_date) >= 7 "
                                             "     THEN year(full_date) + 1 "
                                             "     ELSE year(full_date) END"))
         .withColumn("fiscal_quarter",F.expr("CASE WHEN month(full_date) IN (7,8,9)    THEN 1 "
                                             "     WHEN month(full_date) IN (10,11,12) THEN 2 "
                                             "     WHEN month(full_date) IN (1,2,3)    THEN 3 "
                                             "     ELSE 4 END"))
)

(dim_date
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dim_date"))

print(f"dim_date rows: {spark.table('dim_date').count():,}")


# ================================================================
# STEP 2C: Create the LARGE DIMENSION table (dim_customer)
#         Target size: ~1.5 GB  (Sort Merge Join candidate)
# ================================================================
print("STEP 2B: Creating dim_customer (large dimension ~1.5 GB)")

dim_customer = (
    spark.range(0, LARGE_DIM_ROWS)
         .withColumnRenamed("id", "customer_id")
         .withColumn("first_name",   F.concat(F.lit("First_"),  F.col("customer_id")))
         .withColumn("last_name",    F.concat(F.lit("Last_"),   F.col("customer_id")))
         .withColumn("email",        F.concat(F.col("customer_id"), F.lit("@demo.com")))
         .withColumn("phone",        F.concat(F.lit("555-"), F.expr("lpad(cast(customer_id % 10000 as string), 4, '0')")))
         .withColumn("address",      F.concat(F.col("customer_id"), F.lit(" Main Street, Suite "), F.col("customer_id")))
         .withColumn("city",         F.expr("CASE WHEN customer_id % 5 = 0 THEN 'New York' "
                                            "     WHEN customer_id % 5 = 1 THEN 'Chicago' "
                                            "     WHEN customer_id % 5 = 2 THEN 'Miami'   "
                                            "     WHEN customer_id % 5 = 3 THEN 'Dallas'  "
                                            "     ELSE 'Seattle' END"))
         .withColumn("state",        F.expr("CASE WHEN customer_id % 5 = 0 THEN 'NY' "
                                            "     WHEN customer_id % 5 = 1 THEN 'IL' "
                                            "     WHEN customer_id % 5 = 2 THEN 'FL' "
                                            "     WHEN customer_id % 5 = 3 THEN 'TX' "
                                            "     ELSE 'WA' END"))
         .withColumn("segment",      F.expr("CASE WHEN customer_id % 3 = 0 THEN 'Consumer' "
                                            "     WHEN customer_id % 3 = 1 THEN 'Corporate' "
                                            "     ELSE 'Home Office' END"))
         .withColumn("loyalty_tier", F.expr("CASE WHEN customer_id % 4 = 0 THEN 'Platinum' "
                                            "     WHEN customer_id % 4 = 1 THEN 'Gold' "
                                            "     WHEN customer_id % 4 = 2 THEN 'Silver' "
                                            "     ELSE 'Bronze' END"))
         .withColumn("signup_date",  F.expr("date_add('2015-01-01', cast(customer_id % 3650 as int))"))
         .withColumn("notes",        F.concat(F.lit("Customer profile record for id "), F.col("customer_id"),
                                              F.lit(" with extended demographic and behavioral attributes.")))
)

(dim_customer
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dim_customer"))

print(f"dim_customer rows: {spark.table('dim_customer').count():,}")


# ================================================================
# STEP 3: Create the FACT table (fact_sales)
#         Target size: 50,000,000 rows
# ================================================================
print("STEP 3: Creating fact_sales (50M rows)")

fact_sales = (
    spark.range(0, FACT_ROWS)
         .withColumnRenamed("id", "sales_id")
         .withColumn("customer_id",  F.expr(f"cast(rand(42) * {LARGE_DIM_ROWS} as bigint)"))
         .withColumn("store_id",     F.expr(f"cast(rand(43) * {SMALL_DIM_ROWS} as bigint)"))
         .withColumn("product_id",   F.expr(f"cast(rand(44) * {PRODUCT_DIM_ROWS} as bigint)"))
         .withColumn("quantity",     F.expr("cast(rand(45) * 10 + 1 as int)"))
         .withColumn("unit_price",   F.expr("round(rand(46) * 500 + 5, 2)"))
         .withColumn("amount",       F.expr("round(quantity * unit_price, 2)"))
         .withColumn("sale_date",    F.expr(f"date_add('2022-01-01', cast(rand(47) * {DATE_DIM_DAYS} as int))"))
)

(fact_sales
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("fact_sales"))

print(f"fact_sales rows: {spark.table('fact_sales').count():,}")

StatementMeta(, 6ea3545c-5a40-406a-b081-502cd08ea53d, 7, Finished, Available, Finished, False)

STEP 1: Creating dim_store (small dimension < 10 MB)
dim_store rows: 500
STEP 2A: Creating dim_product (small dimension < 10 MB)
dim_product rows: 10,000
STEP 2B: Creating dim_date (date dimension < 10 MB)
dim_date rows: 1,826
STEP 2B: Creating dim_customer (large dimension ~1.5 GB)
dim_customer rows: 20,000,000
STEP 3: Creating fact_sales (50M rows)
fact_sales rows: 50,000,000


In [3]:
# ================================================================
# STEP 3: Validate table are created
# ================================================================
spark.sql("SHOW TABLES").show()

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 6, Finished, Available, Finished, False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [7]:
# ================================================================
# STEP 4: Validate table sizes
# ================================================================
print("STEP 4: Validating table sizes")

for tbl in ["dim_store", "dim_product", "dim_date", "dim_customer", "fact_sales"]:
    details = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{tbl}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = details["sizeInBytes"] / (1024 * 1024)
    print(f"{tbl:15s} -> {size_mb:10.2f} MB  ({details['numFiles']} files)")

StatementMeta(, 6ea3545c-5a40-406a-b081-502cd08ea53d, 9, Finished, Available, Finished, False)

STEP 4: Validating table sizes
dim_store       ->       0.03 MB  (8 files)
dim_product     ->       0.27 MB  (8 files)
dim_date        ->       0.07 MB  (8 files)
dim_customer    ->     729.71 MB  (8 files)
fact_sales      ->    1025.50 MB  (16 files)


#### <u>**Most useful Spark settings for physical joins**</u>

Notes: This is an advanced session. We strongly recommend relying on Fabric Autotune for anyone who is new to Apache Spark workloads

[Fabric Autotune for Apache Spark](https://learn.microsoft.com/en-us/fabric/data-engineering/autotune?tabs=sparksql)

- Autotune automatically adjusts Apache Spark configurations to reduce workload execution time and improve performance. 
- It helps you avoid manual tuning, which typically requires repeated trial and error. Autotune uses historical execution data from your workloads to iteratively discover and apply effective settings for each workload.

In [1]:
# View current Spark settings
# print(spark.sparkContext.getConf().getAll())
# Prefer sort-merge join when broadcast is not used
print("spark.sql.join.preferSortMergeJoin: ", spark.conf.get("spark.sql.join.preferSortMergeJoin"))

# Max table size eligible for broadcast join
print("autoBroadcastJoinThreshold (MB):", int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold")) // (1024*1024))

# Number of shuffle partitions used during wide transformations
print("spark.sql.shuffle.partitions: ", spark.conf.get("spark.sql.shuffle.partitions"))

# Maximum size of each input file partition during reads
print("maxPartitionBytes (MB):", int(spark.conf.get("spark.sql.files.maxPartitionBytes")) // (1024*1024))


StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 3, Finished, Available, Finished, False)

spark.sql.join.preferSortMergeJoin:  true
autoBroadcastJoinThreshold (MB): 25
spark.sql.shuffle.partitions:  200
maxPartitionBytes (MB): 512


# **Begin analyzing the Spark physical join plan.**

#### **<u>BROADCAST HASH JOIN (small dim < 10 MB threshold)</u>**

- Spark uses Broadcast Hash Join when the smaller side is under spark.sql.autoBroadcastJoinThreshold (default 10 MB).

```PYSPARK
print("STEP 5: Sample queries to demonstrate each physical join")

# --- 5a. BROADCAST HASH JOIN (small dim < 10 MB threshold) ---
print("\n[5a] BROADCAST HASH JOIN")
q1 = spark.sql("""
    SELECT s.region, SUM(f.amount) AS total_sales
    FROM   fact_sales f
    JOIN   dim_store  s ON f.store_id = s.store_id
    GROUP BY s.region
""")
q1.explain(mode="formatted")


# --- 5a2. BROADCAST HASH JOIN with dim_product (also small enough) ---
print("\n[5a2] BROADCAST HASH JOIN (dim_product)")
q1b = spark.sql("""
    SELECT p.category, SUM(f.amount) AS total_sales
    FROM   fact_sales  f
    JOIN   dim_product p ON f.product_id = p.product_id
    GROUP BY p.category
""")
q1b.explain(mode="formatted")


# --- 5a3. BROADCAST HASH JOIN with dim_date + Partition Pruning candidate ---
print("\n[5a3] BROADCAST HASH JOIN (dim_date) with date filter")
q1c = spark.sql("""
    SELECT d.year, d.quarter, SUM(f.amount) AS total_sales
    FROM   fact_sales f
    JOIN   dim_date   d ON f.sale_date = d.full_date
    WHERE  d.year = 2024
    GROUP BY d.year, d.quarter
    ORDER BY d.year, d.quarter
""")
q1c.explain(mode="formatted")
```


In [7]:
# get the file size

print("STEP 4: Validating table sizes")

for tbl in ["dim_store", "fact_sales"]:
    details = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{tbl}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = details["sizeInBytes"] / (1024 * 1024)
    print(f"{tbl:15s} -> {size_mb:10.2f} MB  ({details['numFiles']} files)")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 10, Finished, Available, Finished, False)

STEP 4: Validating table sizes
dim_store       ->       0.03 MB  (8 files)
fact_sales      ->    1025.50 MB  (16 files)


In [8]:
# --- 5a. BROADCAST HASH JOIN (small dim < 10 MB threshold) ---
print("\n[5a] BROADCAST HASH JOIN")
q1 = spark.sql("""
    SELECT s.region, SUM(f.amount) AS total_sales
    FROM   fact_sales f
    JOIN   dim_store  s ON f.store_id = s.store_id
    GROUP BY s.region
""")

q1.show(3)

# Shows the plan.
q1.explain(mode="formatted")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 11, Finished, Available, Finished, False)


[5a] BROADCAST HASH JOIN
+------+--------------------+
|region|         total_sales|
+------+--------------------+
| South|1.753365129352011...|
|  East|1.753807399769984...|
|  West|1.752793001256992E10|
+------+--------------------+
only showing top 3 rows
== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Project (5)
            +- BroadcastHashJoin Inner BuildRight (4)
               :- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- BroadcastExchange (3)
                  +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_store (2)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [2]: [store_id#636L, amount#640]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.mic

#### **<u>SORT MERGE JOIN (both sides large)</u>**
 - If the table is too large to broadcast, Spark falls back to Sort‑Merge Join (SMJ) when the join is:
   - an equijoin (e.g., =),
   - on sortable keys,
   - and both sides are large.

In [9]:
print("STEP 4: Validating table sizes")

for tbl in ["dim_customer", "fact_sales"]:
    details = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{tbl}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = details["sizeInBytes"] / (1024 * 1024)
    print(f"{tbl:15s} -> {size_mb:10.2f} MB  ({details['numFiles']} files)")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 14, Finished, Available, Finished, False)

STEP 4: Validating table sizes
dim_customer    ->     729.71 MB  (8 files)
fact_sales      ->    1025.50 MB  (16 files)


In [10]:
# --- 5b. SORT MERGE JOIN (both sides large) ---
print("\n[5b] SORT MERGE JOIN")
q2 = spark.sql("""
    SELECT c.segment, SUM(f.amount) AS total_sales
    FROM   fact_sales   f
    JOIN   dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.segment
""")
q2.explain(mode="formatted")

q2.show(3)

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 16, Finished, Available, Finished, False)


[5b] SORT MERGE JOIN
== Physical Plan ==
AdaptiveSparkPlan (12)
+- HashAggregate (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Project (8)
            +- SortMergeJoin Inner (7)
               :- Sort (3)
               :  +- Exchange (2)
               :     +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- Sort (6)
                  +- Exchange (5)
                     +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_customer (4)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [2]: [customer_id#1010L, amount#1015]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<customer_id:bigint,amount:double>

(2) Exchange
Inpu

##### Forcing Broadcast Hash Join Types in Spark
- If you understand your Spark environment, you can sometimes force a specific join type.  
- In this case, you have the ability to force a broadcast join instead of letting Spark fall back to 

In [11]:
# --- 5b. SORT MERGE JOIN (both sides large) ---
print("\n[5b] FORCE BROADCAST HASH JOIN")
q2 = spark.sql("""
    SELECT /*+ BROADCAST(c) */ c.segment, SUM(f.amount) AS total_sales
    FROM   fact_sales   f
    JOIN   dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.segment
""")
q2.explain(mode="formatted")

q2.show(3)

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 18, Finished, Available, Finished, False)


[5b] FORCE BROADCAST HASH JOIN
== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Project (5)
            +- BroadcastHashJoin Inner BuildRight (4)
               :- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- BroadcastExchange (3)
                  +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_customer (2)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [2]: [customer_id#1122L, amount#1127]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<customer_id:bigint,amount:double>

(2) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj

#### **<u>SHUFFLE HASH JOIN</u>**

<u>A Shuffle Hash Join (SHJ) sits between:</u>
 - Broadcast Hash Join (fastest)
 - Sort‑Merge Join (most stable)


<u>It is used when:</u>
 - The smaller side is too large to broadcast,
 - But the join does not require the full SMJ pipeline (sort + merge),
 - And the join keys are equijoins (=).

 
<u> Why Spark rarely chooses Shuffle Hash Join:</u>
 - SMJ is more memory‑stable
 - SHJ requires building hash tables in memory
 - If there is data skew, SHJ can cause OOM (out‑of‑memory)
 - SMJ can spill to disk safely → more predictable


Broadcast Hash Join → Sort‑Merge Join → Shuffle Hash Join (last resort)


In [12]:
print("STEP 4: Validating table sizes")

for tbl in ["dim_customer", "fact_sales"]:
    details = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{tbl}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = details["sizeInBytes"] / (1024 * 1024)
    print(f"{tbl:15s} -> {size_mb:10.2f} MB  ({details['numFiles']} files)")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 20, Finished, Available, Finished, False)

STEP 4: Validating table sizes
dim_customer    ->     729.71 MB  (8 files)
fact_sales      ->    1025.50 MB  (16 files)


In [13]:
# get the currect setting value

# Prefer sort-merge join when broadcast is not used
print("spark.sql.join.preferSortMergeJoin: ", spark.conf.get("spark.sql.join.preferSortMergeJoin"))

# Max table size eligible for broadcast join
print("autoBroadcastJoinThreshold (MB):", int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold")) // (1024*1024))

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 21, Finished, Available, Finished, False)

spark.sql.join.preferSortMergeJoin:  true
autoBroadcastJoinThreshold (MB): 25


###### **Shuffle Hash Join is situational and conditions must be met, otherwise Spark will ignore it.**
- **Changing these settings will not force Spark to use a <u>Shuffle Hash Join</u>.**

<u>Why Shuffle Hash Join Was Not Chosen:</u>
 - Setting **autoBroadcastJoinThreshold = -1** also disables Shuffle Hash Join. Spark uses that same threshold internally to decide whether the smaller side is small enough to build an in-memory hash table per partition. When you set it to -1, Spark disables both Broadcast Hash Join and Shuffle Hash Join eligibility, leaving only Sort Merge Join.
 - Even with **preferSortMergeJoin=false**, Spark still requires the smaller side to be roughly 3× smaller than the larger side. In your query, fact_sales (50 M rows) and dim_customer (20 M rows) are close in scale, so Spark does not consider one clearly smaller than the other.
 - **AQE** can also convert Shuffle Hash Join back to Sort Merge Join at runtime if it decides Sort Merge is safer for memory stability.

In [14]:
print("\n[5c] SHUFFLE HASH JOIN (forced)")

spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 22, Finished, Available, Finished, False)


[5c] SHUFFLE HASH JOIN (forced)


In [15]:
# --- 5c. SHUFFLE HASH JOIN (force it by disabling sort merge preference) ---
# Will not work as it will select Sort Merge Join
q3 = spark.sql("""
    SELECT c.loyalty_tier, COUNT(*) AS orders
    FROM   fact_sales   f
    JOIN   dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
""")
q3.explain(mode="formatted")

q3.show(3)

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 24, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (12)
+- HashAggregate (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Project (8)
            +- SortMergeJoin Inner (7)
               :- Sort (3)
               :  +- Exchange (2)
               :     +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- Sort (6)
                  +- Exchange (5)
                     +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_customer (4)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [1]: [customer_id#1336L]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<customer_id:bigint>

(2) Exchange
Input [1]: [customer_id#1336L]
Arguments: hashpartiti

##### Optional and Important Notes
- Because AQE can override join choices at runtime, you can temporarily disable it before running the query or `explain` to increase the chance that the physical plan shows `ShuffledHashJoin`. However, this is not guaranteed—Spark may still fall back to `SortMergeJoin` if Shuffle Hash Join conditions are not met.

##### Key Takeaway
Shuffle Hash Join is rarely selected in real Spark workloads because it is significantly less memory‑stable than Sort Merge Join.

In [16]:
# Disable AQE temporarily so the physical plan shows exactly what the optimizer chose
spark.conf.set("spark.sql.adaptive.enabled", "false")

q3 = spark.sql(f"""
    SELECT c.loyalty_tier, COUNT(*) AS orders
    FROM   {SCHEMA}.fact_sales   f
    JOIN   {SCHEMA}.dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
""")
q3.explain(mode="formatted")

q3.show(3)

# Re-enable AQE for the rest of the demo
spark.conf.set("spark.sql.adaptive.enabled", "true")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 27, Finished, Available, Finished, False)

== Physical Plan ==
* HashAggregate (13)
+- Exchange (12)
   +- * HashAggregate (11)
      +- * Project (10)
         +- * SortMergeJoin Inner (9)
            :- * Sort (4)
            :  +- Exchange (3)
            :     +- * ColumnarToRow (2)
            :        +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
            +- * Sort (8)
               +- Exchange (7)
                  +- * ColumnarToRow (6)
                     +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_customer (5)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [1]: [customer_id#1377L]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<customer_id:bigint>

(2) ColumnarToRow [c

In [17]:
# Reset defaults
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 30, Finished, Available, Finished, False)

##### **Force Hash Join Condition with Hint**

In [18]:
q3 = spark.sql(f"""
    SELECT /*+ SHUFFLE_HASH(c) */
           c.loyalty_tier, COUNT(*) AS orders
    FROM   {SCHEMA}.fact_sales   f
    JOIN   {SCHEMA}.dim_customer c ON f.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
""")
q3.explain(mode="formatted")

q3.show(3)

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 31, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- ShuffledHashJoin Inner BuildRight (5)
               :- Exchange (2)
               :  +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- Exchange (4)
                  +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_customer (3)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [1]: [customer_id#1418L]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<customer_id:bigint>

(2) Exchange
Input [1]: [customer_id#1418L]
Arguments: hashpartitioning(customer_id#1418L, 200), ENSURE_REQUIREMENTS, [p

#### <u>**BROADCAST NESTED LOOP JOIN**</u>

<u>Spark uses a Broadcast Nested Loop Join when:</u>
 - The join condition is non‑equi (e.g., >=, <=, !=, BETWEEN, etc.)
 - The query performs a cross join or Cartesian product
 - One table is small enough to broadcast, and the other is large

<u>Execution steps</u>
 - Broadcast the smaller table
 - Nested loop over the larger table


<u>Performance side effects:</u>
 - Extremely expensive: O(N × M) nested‑loop comparisons
 - Very high CPU usage because Spark cannot use hashing or sorting
 - Heavy memory pressure on executors due to broadcasting
 - Poor parallelism scaling — more executors = more nested loops, not less work
 - Highly vulnerable to data skew, which can cause OOM or long‑running tasks

In [19]:
# --- 5d. BROADCAST NESTED LOOP JOIN (non-equi join) ---
print("\n[5d] BROADCAST NESTED LOOP JOIN")
q4 = spark.sql("""
    SELECT s.region, COUNT(*) AS cnt
    FROM   fact_sales f
    JOIN   dim_store  s ON f.store_id >= s.store_id
    GROUP BY s.region
""")
q4.explain(mode="formatted")

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 33, Finished, Available, Finished, False)


[5d] BROADCAST NESTED LOOP JOIN
== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Project (5)
            +- BroadcastNestedLoopJoin Inner BuildRight (4)
               :- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales (1)
               +- BroadcastExchange (3)
                  +- Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_store (2)


(1) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.fact_sales
Output [1]: [store_id#1460L]
Batched: true
Location: PreparedDeltaFileIndex [abfss://4930f931-cbdb-4275-a8f2-075de4bb5ec8@onelake.dfs.fabric.microsoft.com/46a02d2e-11c4-444b-8450-1ca356f22a3c/Tables/optimization/fact_sales]
ReadSchema: struct<store_id:bigint>

(2) Scan parquet spark_catalog.chimcobldhq2aj39epiiqh35dlniqhbeeoikoi1g64imus3kd5mmiuj1ehkmurg.dim_store
Output [2]

#### **bucketing** 
- A bucketed join requires both tables to be bucketed on the same join key and typically the same number of buckets. Sorting within buckets is optional but beneficial, and partitioning is completely independent of bucketing.

<u>Why Bucketing Requires a Table:</u>
 - Bucketing **is a physical storage layout**, not a runtime optimization. Spark must write the data to disk in a specific way, distributing rows across a fixed number of bucket files based on a hash of the bucket column. That layout only exists if the table has been created and written using bucketBy(...).
 - The bucketing metadata is stored in the metastore. When Spark reads a bucketed table, it looks up the metastore to learn how many buckets exist, which columns were used for bucketing, and whether the data was sorted within each bucket. Without a registered table, that metadata does not exist, so Spark cannot skip the shuffle.
 - The query planner needs a stable reference to the layout. Bucketing works because Spark trusts that the physical files match the declared bucket configuration. This trust requires a persisted, catalog-registered table. A DataFrame alone has no metastore entry Spark can rely on.


In [20]:
# Rewrite dim_customer and fact_sales tables

SCHEMA = "optimization"

# Reload from Lakehouse so this cell can run independently
fact_sales   = spark.table(f"{SCHEMA}.fact_sales")
dim_customer = spark.table(f"{SCHEMA}.dim_customer")

# Note: bucketBy is not supported when writing Delta format tables in Fabric.
# Use parquet format for bucketed tables so bucketing metadata is honored.
(fact_sales
    .write
    .mode("overwrite")
    .format("parquet")
    .bucketBy(16, "customer_id")
    .sortBy("customer_id")
    .saveAsTable(f"{SCHEMA}.fact_sales_bucketed"))

(dim_customer
    .write
    .mode("overwrite")
    .format("parquet")
    .bucketBy(16, "customer_id")
    .sortBy("customer_id")
    .saveAsTable(f"{SCHEMA}.dim_customer_bucketed"))

StatementMeta(, 5e681577-2c95-4df6-b3dd-dd4e206e4977, 34, Finished, Available, Finished, False)

In [ ]:
print("STEP 4: Validating table sizes")

for tbl in ["dim_customer", "fact_sales"]:
    details = spark.sql(f"DESCRIBE DETAIL {SCHEMA}.{tbl}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = details["sizeInBytes"] / (1024 * 1024)
    print(f"{tbl:15s} -> {size_mb:10.2f} MB  ({details['numFiles']} files)")

**Focus on what disappeared: **
 - The Exchange hashpartitioning operator on the join key. That single missing step represents the shuffle of 70 million rows that bucketing eliminated. 
 - The join algorithm did not change, but the amount of work Spark had to do before the join was reduced dramatically. 
 - This is why bucketing was historically used to accelerate repeated large-table joins, and it is also why Microsoft now recommends Liquid Clustering as the modern replacement in Fabric Runtime 2.0.

In [ ]:
# --- 5e. BUCKETED JOIN (pre-shuffled data) ---
print("\n[5e] BUCKETED JOIN (write bucketed tables, then join)")


q5 = spark.sql(f"""
    SELECT c.segment, SUM(f.amount) AS total_sales
    FROM   {SCHEMA}.fact_sales_bucketed   f
    JOIN   {SCHEMA}.dim_customer_bucketed c ON f.customer_id = c.customer_id
    GROUP BY c.segment
""")
q5.explain(mode="formatted")

#### <u>**JOIN ORDER MATTERS**</u>
 - Join order matters because Spark processes joins one at a time, and each intermediate result becomes the input to the next join. If you carry too many rows into the next stage, you multiply the cost of every downstream operation: shuffle, sort, memory, network, and CPU.
 
<u>Which plan is the most expensive?</u>
 - The first plan is more expensive: (Plan with: BroadcastHashJoin + SortMergeJoin inside it)
 - The second plan is less expensive: because it uses a BroadcastHashJoin first, then only one SortMergeJoin later — a cheaper sequence.

In [ ]:
# ================================================================
# STEP 6: JOIN ORDER MATTERS
# Demonstrates the impact of joining selective dimensions first
# ================================================================
print("STEP 6: Demonstrating Join Order Impact")

SCHEMA = "optimization"

# Disable AQE and Catalyst join reordering so we can see the raw effect
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.cbo.enabled", "false")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "false")


# ---------------------------------------------------------------
# 6a. BAD JOIN ORDER
# Join the two large tables first (fact_sales x dim_customer),
# then join the small dimension (dim_store) with a very selective filter.
# ---------------------------------------------------------------
print("\n[6a] BAD JOIN ORDER (large x large first, filter last)")

q_bad = spark.sql(f"""
    SELECT s.region, c.segment, SUM(f.amount) AS total_sales
    FROM   {SCHEMA}.fact_sales   f
    JOIN   {SCHEMA}.dim_customer c ON f.customer_id = c.customer_id
    JOIN   {SCHEMA}.dim_store    s ON f.store_id    = s.store_id
    WHERE  s.region = 'North'
    GROUP BY s.region, c.segment
""")
q_bad.explain(mode="formatted")

import time
start = time.time()
q_bad.collect()
print(f"[6a] Bad join order runtime: {time.time() - start:.2f} seconds")


# ---------------------------------------------------------------
# 6b. GOOD JOIN ORDER
# Join the small selective dimension (dim_store) first,
# apply the filter early, then join the larger dim_customer table.
# ---------------------------------------------------------------
print("\n[6b] GOOD JOIN ORDER (selective filter first)")

q_good = spark.sql(f"""
    SELECT s.region, c.segment, SUM(f.amount) AS total_sales
    FROM   (SELECT store_id, region
            FROM   {SCHEMA}.dim_store
            WHERE  region = 'North')       s
    JOIN   {SCHEMA}.fact_sales   f ON f.store_id    = s.store_id
    JOIN   {SCHEMA}.dim_customer c ON f.customer_id = c.customer_id
    GROUP BY s.region, c.segment
""")
q_good.explain(mode="formatted")

start = time.time()
q_good.collect()
print(f"[6b] Good join order runtime: {time.time() - start:.2f} seconds")


# ---------------------------------------------------------------
# Restore defaults
# ---------------------------------------------------------------
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")

print("\nJoin order demo complete. Compare the two runtimes and physical plans.")